**This part is required for all the notebook**

In [8]:
import os
import subprocess
import threading
import serial.tools.list_ports
import ipywidgets as widgets
from IPython.display import display, clear_output


def run_process_cmd(cmd=[]):
    process =  subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True
    )

    buffer = ""
    while True:
        line = process.stdout.readline()
        if not line and process.poll() is not None:
            break
        if line:
            buffer += line
            clear_output(wait=True)
            print(buffer)

def detect_serial_ports(_):
    available_ports = []

    ports = serial.tools.list_ports.comports()
    for port in ports:
        if any(x in port.device for x in ["ttyUSB", "ttyACM", "cu.usbmodem", "usbserial", "COM"]):
            available_ports.append(port.device)

    return available_ports


# NOTE:
This part of the notebook requires internet connection, please run only once.

# Requirements
## Clone or update Catsniffer-Tools
This code cell will execute part of the CatSniffer setup process. It checks out or updates the CatSniffer repository containing firmware and utilities.

> **Please run the following code block before begin the Labs!**

In [21]:


repo_url = "https://github.com/ElectronicCats/CatSniffer-Tools.git"
target_dir = "CatSniffer-Tools"

if not os.path.exists(target_dir):
    print(f"Cloning repository {repo_url}...")
    !git clone {repo_url}
    print("Done")
else:
    print(f"Updating existing repository in {target_dir}...")
    try:
        run_process_cmd(["git", "-C", target_dir, "fetch"])
        run_process_cmd(["git", "-C", target_dir, "pull"])
    except subprocess.CalledProcessError as e:
        print(f"Failed to update repository: {e}")

Already up to date.



## Installing the Requierements
This code cell will execute part of the CatSniffer Catnip tool to download the last firmware from the repo.

In [40]:
# Sanity check
req_file = "requirements.txt"
if not os.path.exists(req_file):
    print("Please, contact with the trainers.")
else:
    print("Installing the requirements...")
    run_process_cmd(["pip", "install", "-r", req_file])

  Using cached pyusb-1.3.1-py3-none-any.whl.metadata (2.5 kB)
Using cached pyusb-1.3.1-py3-none-any.whl (58 kB)

[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip



## Download Firmware files
This code cell will execute part of the CatSniffer Catnip tool process. It checks out and download the CatSniffer firmware.

In [28]:
catnip_path = os.path.join(target_dir, "catnip_uploader")
cmd = ["python", os.path.join(catnip_path, "catnip_uploader.py"), "releases"]

# Sanity check
if not os.path.exists(catnip_path):
    print("Please, run the first code block.")
else:
    print("Downloading the firmware...")
    run_process_cmd(cmd)

[WARNING] No releases found. Fetching from GitHub.

[SUCCESS] Firmware 12/12 downloaded.
[INFO] Using local releases: C:\Users\skinf\OneDrive\Escritorio\CatSniffer\workshops\jupyter-notebooks\CatSniffer-Tools\catnip_uploader/releases_board-v3.x-v1.2.2 with tag version: board-v3.x-v1.2.2
                                   Releases                                    
+-----------------------------------------------------------------------------+
| Firmware                    | Microcontroller | Description                 |
|-----------------------------+-----------------+-----------------------------|
| sniffle_cc1352p7_1m.hex     | CC1352          | BLE sniffer for Bluetooth 5 |
|                             |                 | and 4.x (LE) from NCC       |
|                             |                 | Group. See                  |
|                             |                 | [Sniffle](https://github.c… |
|                             |                 | (Windows/Linux/Mac)   

## Load the utility functions

In [2]:
import time
import subprocess
from pathlib import Path

target_dir = "CatSniffer-Tools"

catnip_path = os.path.join(target_dir, "catnip_uploader")

# Sanity check
def get_release_folder():
    dir_files = os.listdir(catnip_path)
    for dirf in dir_files:
        if dirf.startswith("releases_"):
            return dirf
    return None

def flash_uf2_firmware(uf2_file):
    file_path = Path(uf2_file)
    while True:
        cmd = [
            "powershell",
            "-Command",
            "(Get-WmiObject Win32_Volume | Where-Object { $_.Label -match 'RPI-RP2' }).DriveLetter"
        ]
        drive = subprocess.run(cmd, capture_output=True, text=True).stdout.strip()
        if drive:
            print(f"Board detected ({drive}). Flashing firmware...")
            cmd = [
                "powershell",
                "-Command",
                f"""
                $drive = (Get-WmiObject Win32_Volume | Where-Object {{ $_.Label -match 'RPI-RP2' }}).DriveLetter
                if ($drive) {{
                    Copy-Item "{file_path.resolve()}" "$drive\\"
                    Write-Host "File flashed to $drive"
                }} else {{
                    Write-Host "RP2040 not found"
                }}
                """
            ]
            
            print(subprocess.run(cmd, capture_output=True, text=True).stdout)
            break
        print("Waiting connection RP2040...")
        time.sleep(2)

---

# Hands-On Lab #1: Catsniffer + Meshtastic

## Objectives
- Explain the role of CatSniffer in capturing and analyzing 915 MHz RF signals.
- Configure CatSniffer as a LoRa sniffer for Meshtastic traffic.
- Identify and isolate Meshtastic communication patterns in the RF spectrum.
- Use the Meshtastic decrypter with known PSKs to extract public messages.
- Use the Meshtastic live decoder to parse and understand packet payloads.

## Challenge
Flash the CatSniffer with the LoRa sniffer firmware. Monitor real-time activity in the 900 MHz band to identify Meshtastic node transmissions.
Distinguish Meshtastic packets from other traffic based on RF characteristics.
Capture encrypted payloads, apply known keys using the Meshtastic decrypter, and extract readable messages. Use the live decoder to analyze packet structure and content.

## Walk-Through

First of all we need to flash our LoRa Sniffer for the RP2040 since this is the microcontroller who has access to the SX1262 by SPI.
![Catsniffer Components Diagram](static/block_diagrams.png)

### Flash firmware

> Run the next code, and after that follow the steps to put the bootloader mode


In [49]:
serialpass_file = "lorasniffer.uf2"
release_path = get_release_folder()
if release_path is None:
    print("Error getting releases folder, please check the requirements blocks")
else:
    uf2_file = os.path.join(catnip_path, release_path, serialpass_file)
    flash_uf2_firmware(uf2_file)


Waiting connection RP2040...
Waiting connection RP2040...
Waiting connection RP2040...
Waiting connection RP2040...
Board detected (D:). Flashing firmware...
File flashed to D:




### RP2040 bootloader mode:

![Catsniffer Board](static/banner.jpg)


- Press and hold boot RP2040
- While Holding Boot RP2040, press Reset RP2040
- The device will be shown as a thumbdrive

![RP 2040 Boot](static/RP_USB.png)



For this laboratory we are going to try to catch messages from the meshtastic firmware for Black Hat EU. If we dig a bit in the firmware repository for meshtastic firmware we can find the jsonc file used to configure any node once programmed.

### Radio Configuration

Radio configurations for this meshtastic network are:

- Frequency = 917.25 MHz
- Bandwidth = 500 Khz 
- Spreading Factor = 7
- Coding Rate = 4/5
- Sync Word = 0x2B
- Preamble Length = 16

If these radio configurations are not equal, you wont be able to receive any data from this mesh.

### Configuring the radio
- **Configuration commands**:
    - set_rx
    - set_tx
    - set_tx_hex
    - set_tx_ascii
    - set_region
    - set_chann
    - set_freq
    - set_sf
    - set_bw
    - set_cr
    - set_sw
    - set_pl
    - set_op
- **Monitor commands**:
    - get_freq
    - get_sf
    - get_bw
    - get_cr
    - get_sw
    - get_pl
    - get_op
    - get_config

We use the commands to setup the radio:
- `set_freq 917.25`
- `set_bw 9`
- `set_pl 16`
- `set_sw 0x2B`
- `set_rx`

In [70]:
import serial
import time

available_ports = []
serial_conn = None
serial_loop_reading = False
th = threading.Thread()
# GUI
output_serial = widgets.Output(layout={'border': '1px solid black'})
btn_scan_ports = widgets.Button(description="Scan ports")
text_user_command = widgets.Text(placeholder="Type the command", layout=widgets.Layout(width='25%'))
btn_send_command = widgets.Button(description="Send", icon="arrow-right")
btn_clear_output = widgets.Button(description="Clear console", icon="eraser")
btn_loop_read = widgets.Button(description="Open", icon="play-circle", button_style="success")
dropdown_ports = widgets.Dropdown(options=available_ports, description='Ports:', layout=widgets.Layout(width='20%'))

def connect_serial():
    global serial_conn
    try:
        serial_conn = serial.Serial(dropdown_ports.value, 921600, timeout=1)
        return True
    except Exception as e:
        return False

def send_command_string(command:str):
    if serial_conn and serial_conn.is_open:
        cmd = command+'\r\n'
        serial_conn.write(cmd.encode())

def send_command_string_with_response(command:str):
    global serial_conn

    if serial_conn and serial_conn.is_open:
        cmd = command+'\r\n'
        serial_conn.write(cmd.encode())
        time.sleep(0.3)
        try: 
            resp = serial_conn.read_all().decode(errors='ignore')
            return resp
        except:
            return 'Nan'
            
    else:
        return 'Nan'

def disconnect_serial():
    global serial_conn
    
    if serial_conn and serial_conn.is_open:
        serial_conn.close()
        serial_conn = None
        return True
    else: 
        return False

def loop_reading():
    global serial_loop_reading
    global serial_conn

    if connect_serial():
        while serial_loop_reading:
            data = serial_conn.read_all()
            if data:
                show_prompt_catsniffer(data.decode())
            time.sleep(0.1)
        disconnect_serial()

def show_prompt_catsniffer(data):
    prompt = f"\n[CATSNIFFER] {data}"
    output_serial.append_stdout(prompt)

def show_prompt_user(data):
    prompt = f"> {data}\n"
    output_serial.append_stdout(prompt)

def on_send_command(_):
    cmd = text_user_command.value
    send_command_string(cmd)
    text_user_command.value = ""

def on_clear_output(_):
    output_serial.clear_output()

def on_scan_port(_):
    dropdown_ports.options = detect_serial_ports(None)

def on_loop_reading(_):
    global serial_loop_reading
    global th

    serial_loop_reading = not serial_loop_reading
    
    if serial_loop_reading:
        if not th.is_alive():
            th = threading.Thread(target=loop_reading)
            th.start()
            btn_loop_read.description = "Close"
            btn_loop_read.icon="stop-circle"
            btn_loop_read.button_style='danger'
    else:
        btn_loop_read.description = "Open"
        btn_loop_read.icon="play-circle"
        btn_loop_read.button_style='success'

btn_scan_ports.on_click(on_scan_port)
btn_send_command.on_click(on_send_command)
btn_clear_output.on_click(on_clear_output)
btn_loop_read.on_click(on_loop_reading)

on_scan_port(None)

display(widgets.VBox([
    widgets.Box([btn_scan_ports, dropdown_ports, btn_loop_read]),
    # label_port,detect_minino_button, output_serial
    widgets.HBox([text_user_command, btn_send_command, btn_clear_output]), 
    #widgets.Box([command_input, send_button]), 
    output_serial
]))